# Pipeline — Paso 5: Análisis del dataset de anotaciones

Analiza las anotaciones de las 3 herramientas (Label Studio, CVAT, ELAN), calcula estadísticas de calidad y genera el CSV unificado.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.config import DATA_DIR, CVAT_ANOTACIONES_DIR, ELAN_ANOTACIONES_DIR
from utils.adaptadores import (cargar_label_studio, cargar_cvat, cargar_elan)

# ─── Rutas por fuente ────────────────────────────────────────────────────────
CSV_LS           = os.path.join(DATA_DIR, "anotaciones", "label_studio", "dataset_etiquetas_metadatos.csv")
# Fuentes originales (XML / EAF) — definidas en utils/config.py

CARPETA_CVAT_XML = CVAT_ANOTACIONES_DIR
CARPETA_ELAN_EAF = ELAN_ANOTACIONES_DIR


# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISIS
# ══════════════════════════════════════════════════════════════════════════════

def seccion(titulo: str):
    print(f"\n--- {titulo} ---")


def analizar_herramienta(df: pd.DataFrame, nombre: str):
    """Muestra estadísticas de calidad para un DataFrame normalizado."""
    print(f"\n{'='*55}")
    print(f"  {nombre.upper()}  ({len(df)} videos | {df['glosa'].nunique()} señas)")
    print(f"{'='*55}")

    seccion("DURACIÓN DEL SEGMENTO")
    d = df["duracion_seg_etiqueta"]
    print(f"  Promedio: {d.mean():.3f}s  |  min: {d.min():.3f}s  |  max: {d.max():.3f}s")
    print(f"  < 0.3s:   {(d < 0.3).sum()}  |  < 0.5s: {(d < 0.5).sum()}  |  >= 0.5s: {(d >= 0.5).sum()}")

    if "confianza" in df.columns and df["confianza"].notna().any():
        seccion("CONFIANZA")
        c = df["confianza"]
        print(f"  Promedio: {c.mean():.3f}")
        print(f"  < 0.3: {(c < 0.3).sum()}  |  0.3-0.5: {((c>=0.3)&(c<0.5)).sum()}  |  >= 0.5: {(c>=0.5).sum()}")

    if "blur_score" in df.columns and df["blur_score"].notna().any():
        seccion("NITIDEZ (blur_score)")
        b = df["blur_score"]
        print(f"  Promedio: {b.mean():.2f}  |  < 20: {(b<20).sum()}  |  >= 50: {(b>=50).sum()}")

    if "keypoints_validos" in df.columns and df["keypoints_validos"].notna().any():
        seccion("KEYPOINTS VÁLIDOS")
        k = df["keypoints_validos"]
        print(f"  Promedio: {k.mean():.3f}  |  < 0.3: {(k<0.3).sum()}  |  >= 0.5: {(k>=0.5).sum()}")

    if "mano_dominante" in df.columns and df["mano_dominante"].notna().any():
        seccion("MANO DOMINANTE")
        print(df["mano_dominante"].value_counts().to_string())


def comparativa(frames: dict[str, pd.DataFrame]):
    """Tabla resumen comparando duración media y cobertura entre herramientas."""
    if len(frames) < 2:
        return
    print(f"\n{'='*55}")
    print("  COMPARATIVA ENTRE HERRAMIENTAS")
    print(f"{'='*55}")
    print(f"{'Herramienta':<20} {'Videos':>7} {'Señas':>6} {'Dur.media':>10} {'Dur.std':>8}")
    print("-" * 55)
    for nombre, df in frames.items():
        d = df["duracion_seg_etiqueta"]
        print(f"{nombre:<20} {len(df):>7} {df['glosa'].nunique():>6} "
              f"{d.mean():>9.3f}s {d.std():>7.3f}s")

    # Diferencia de duración en videos compartidos (mismo video_id + glosa)
    nombres = list(frames.keys())
    for i in range(len(nombres)):
        for j in range(i + 1, len(nombres)):
            a, b = nombres[i], nombres[j]
            merge = frames[a][["video_id", "glosa", "duracion_seg_etiqueta"]].merge(
                frames[b][["video_id", "glosa", "duracion_seg_etiqueta"]],
                on=["video_id", "glosa"], suffixes=(f"_{a}", f"_{b}")
            )
            if merge.empty:
                continue
            diff = merge[f"duracion_seg_etiqueta_{b}"] - merge[f"duracion_seg_etiqueta_{a}"]
            print(f"\n  Diferencia duracion {b} - {a}:")
            print(f"    videos compartidos: {len(merge)}")
            print(f"    media: {diff.mean():+.3f}s  |  std: {diff.std():.3f}s  "
                  f"|  max: {diff.max():+.3f}s  |  min: {diff.min():+.3f}s")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 55)
    print("ANÁLISIS DEL DATASET — LSEc")
    print("Detectando fuentes disponibles...")
    print("=" * 55)

    # — Label Studio —
    df_ls = cargar_label_studio(CSV_LS)

    # — CVAT: usa annotations.xml de anotaciones/cvat/ (proyecto autocontenido) —
    df_cvat = cargar_cvat(CARPETA_CVAT_XML)
    if df_cvat is None or len(df_cvat) == 0:
        print("  [AVISO] CVAT: annotations.xml no encontrado en", CARPETA_CVAT_XML)

    # — ELAN: usa archivos .eaf de anotaciones/elan/ (proyecto autocontenido) —
    df_elan = cargar_elan(CARPETA_ELAN_EAF)
    if df_elan is None or len(df_elan) == 0:
        print("  [AVISO] ELAN: archivos .eaf no encontrados en", CARPETA_ELAN_EAF)

    fuentes = {
        "label_studio": df_ls,
        "cvat":         df_cvat,
        "elan":         df_elan,
    }

    disponibles = {k: v for k, v in fuentes.items() if v is not None and len(v) > 0}

    for nombre, df in fuentes.items():
        if df is not None and len(df) > 0:
            print(f"  [OK] {nombre:<18} {len(df)} registros")
        else:
            print(f"  [--] {nombre:<18} no disponible")

    if not disponibles:
        print("\n[ERROR] No se encontró ninguna fuente de datos.")
        return

    # Análisis individual por herramienta
    for nombre, df in disponibles.items():
        analizar_herramienta(df, nombre)

    # Comparativa cuando hay más de una herramienta
    comparativa(disponibles)

    # Dataset unificado (útil para pasos siguientes)
    unificado = pd.concat(disponibles.values(), ignore_index=True)
    salida = os.path.join(DATA_DIR, "datos", "dataset_anotaciones_unificado.csv")
    unificado.to_csv(salida, index=False, encoding="utf-8-sig")
    print(f"\n{'='*55}")
    print(f"Dataset unificado guardado: {salida}")
    print(f"Total registros: {len(unificado)} | Herramientas: {unificado['herramienta'].nunique()}")

main()


C:\Users\dell\AppData\Local\Temp\ipykernel_20236\4207052761.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


ANÁLISIS DEL DATASET — LSEc
Detectando fuentes disponibles...


  [OK] label_studio       1053 registros
  [OK] cvat               51 registros
  [OK] elan               35 registros

  LABEL_STUDIO  (1053 videos | 528 señas)

--- DURACIÓN DEL SEGMENTO ---
  Promedio: 1.559s  |  min: 0.100s  |  max: 3.804s
  < 0.3s:   27  |  < 0.5s: 68  |  >= 0.5s: 985

--- CONFIANZA ---
  Promedio: 0.564
  < 0.3: 115  |  0.3-0.5: 308  |  >= 0.5: 630

--- NITIDEZ (blur_score) ---
  Promedio: 19.59  |  < 20: 606  |  >= 50: 16

--- KEYPOINTS VÁLIDOS ---
  Promedio: 0.998  |  < 0.3: 0  |  >= 0.5: 1053

--- MANO DOMINANTE ---
mano_dominante
derecha      439
ambas_izq    312
ambas_der    242
izquierda     60

  CVAT  (51 videos | 35 señas)

--- DURACIÓN DEL SEGMENTO ---
  Promedio: 1.144s  |  min: 0.484s  |  max: 2.486s
  < 0.3s:   0  |  < 0.5s: 1  |  >= 0.5s: 50

--- CONFIANZA ---
  Promedio: 1.000
  < 0.3: 0  |  0.3-0.5: 0  |  >= 0.5: 51

--- MANO DOMINANTE ---
mano_dominante
ambas      31
derecha    20

  ELAN  (35 videos | 35 señas)

--- DURACIÓN DEL SEGMENTO ---
  